# Study 9 v2: train C-M (16×128 latent diffusion) on Colab
Run the cells top to bottom. If the session disconnects, run cells 1–4 again and then cell 6: training resumes from the last checkpoint (saved every 1,000 steps to Google Drive).

**Before starting:** Runtime → Change runtime type → **T4 GPU** (or better). Upload `ecg_colab_bundle.zip` to the top level of *My Drive*.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (a pop-up asks for permission)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Unpack the bundle into Drive once (about 2 min); later runs skip this
import os, zipfile
BUNDLE = '/content/drive/MyDrive/ecg_colab_bundle.zip'
WORK = '/content/drive/MyDrive/ECG_colab'
if not os.path.exists(WORK + '/latent_diffusion/train_diffusion_v2.py'):
    zipfile.ZipFile(BUNDLE).extractall(WORK)
%cd {WORK}
!ls

In [ ]:
# 4. Install libraries, test pykeops (in a separate process so the notebook never holds the GPU)
!pip -q install einops opt_einsum pykeops
!python -c "import pykeops; pykeops.test_torch_bindings(); print('pykeops OK')" || echo 'pykeops not usable, slow kernel will be used'
import os
lock = 'latent_diffusion/artifacts/v2/gpu.lock'
if os.path.exists(lock):
    os.remove(lock); print('removed old lock file')

In [ ]:
# 5. Speed test: 1,000 steps. Note the 'min' number on the line starting 'cm_profile 1000'.
!V2_SINGLE_GPU_HOST=1 python -u -m latent_diffusion.train_diffusion_v2 --name cm_profile --layers 16 --width 128 --state 64 --steps 1000 2>&1 | grep --line-buffered -v "pyKeOps\] Warning"

In [ ]:
# 6. MAIN RUN. Rerun this same cell after any disconnect: it resumes.
#    One line every 1,000 steps: name, step, dev loss, per-noise-level loss, elapsed minutes.
STEPS = 100000   # lower only if the speed test says 100k would take more than 12 h
!V2_SINGLE_GPU_HOST=1 python -u -m latent_diffusion.train_diffusion_v2 --name cm --layers 16 --width 128 --state 64 --steps {STEPS} 2>&1 | grep --line-buffered -v "pyKeOps\] Warning"

In [ ]:
# 7. Pack the results for the laptop -> My Drive/cm_results.zip
import zipfile, glob
files = (glob.glob('latent_diffusion/artifacts/v2/diffusion_cm*/*.pt')
         + glob.glob('latent_diffusion/results/v2/diffusion_training_cm*.json')
         + glob.glob('latent_diffusion/audits/v2/phaseCD_cm*'))
with zipfile.ZipFile('/content/drive/MyDrive/cm_results.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in files:
        z.write(f); print(f)
!nvidia-smi --query-gpu=name --format=csv > /content/drive/MyDrive/cm_gpu.txt; cat /content/drive/MyDrive/cm_gpu.txt